# Submission check: Track 1 EEG-to-Image

Loads `dist/submission.zip` **the way Codabench does** (unzipped, read-only, `Solver.load_model(meta)` then `model.predict(X)`), checks the contract, and visualizes the retrieval.

Run `make test-submission` first: it builds `dist/submission.zip` and unzips it into `dist/unzipped/`.

**Data:** the official `Simulated` dataset with the **warm-up input shape** (63 channels × 120 samples → 1536-d embedding). It is synthetic, so the scores here only check that the pipeline works; they say nothing about the real warm-up score.

In [ ]:
import copy
import importlib.util
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TRACK = REPO / "external/2026-competition/tracks/image_decoding"
SUBMISSION_DIR = REPO / "dist/unzipped"
assert (SUBMISSION_DIR / "submission.py").exists(), "Run `make test-submission` first"

# The worker provides `benchmark_utils` from the track folder; do the same here.
sys.path.insert(0, str(TRACK))
from benchopt.utils.run_context import RunContext
from benchmark_utils.metrics import group_means, topk_accuracy


def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def simulated_data(seed="0", n_test=400):
    """Official Simulated dataset with warm-up dims. seed="0" reproduces `benchopt run`'s data."""
    sim = load_module("simulated", TRACK / "datasets/simulated.py")
    ds = sim.Dataset.get_instance(n_chans=63, n_times=120, n_images=50, n_outputs=1536,
                                  n_train=200, n_test=n_test, sfreq=120.0)
    RunContext(base_seed=seed).attach(None, ds, None)
    return ds.get_data()


data = simulated_data()
meta = {k: v for k, v in data.items() if k not in ("train_loader", "test_loader")}
meta["submission_dir"] = SUBMISSION_DIR
print({k: v for k, v in meta.items() if k not in ("ch_names", "chs_info")})

## 1. Load the submission like Codabench

`submission_dir` is made read-only, as on the platform. If `load_model` tries to write there, this cell fails.

In [ ]:
for f in sorted(p for p in SUBMISSION_DIR.iterdir() if p.is_file()):
    print(f"{f.name:<16}{f.stat().st_size / 1e6:8.2f} MB")

os.system(f'chmod -R a-w "{SUBMISSION_DIR}"')
try:
    submission = load_module("submission", SUBMISSION_DIR / "submission.py")
    solver = submission.Solver.get_instance()
    solver.train_loader = None  # inference-only, like the platform
    model = solver.load_model(meta)
finally:
    os.system(f'chmod -R u+w "{SUBMISSION_DIR}"')

n_params = sum(p.numel() for p in model.parameters())
print(f"\nLoaded {type(model).__name__}: {n_params / 1e6:.2f} M parameters, training={model.training}")

## 2. Contract checks

The things that make an upload fail (see "Common submission errors" in the Submission Guide).

In [ ]:
X, y, info = next(iter(data["test_loader"]))
pred = model.predict(X)

source = (SUBMISSION_DIR / "submission.py").read_text()
imports = sorted({line.split()[1].split(".")[0] for line in source.splitlines()
                  if line.startswith(("import ", "from "))})

one_by_one = torch.cat([model.predict(X[i:i + 1]) for i in range(len(X))])

checks = {
    f"output shape {tuple(pred.shape)} == (B, n_outputs)": tuple(pred.shape) == (len(X), meta["n_outputs"]),
    f"output dtype {pred.dtype} is float": pred.dtype.is_floating_point,
    "output is finite (no NaN/Inf)": bool(torch.isfinite(pred).all()),
    f"output on input device ({X.device})": pred.device == X.device,
    "same result for B=1 and B=32 (no batch dependence)": torch.allclose(pred, one_by_one, atol=1e-5),
    "model survives copy.deepcopy": copy.deepcopy(model) is not None,
    f"only worker-safe imports {imports}": set(imports) <= {"pathlib", "torch", "benchmark_utils"},
}
for name, ok in checks.items():
    print("PASS" if ok else "FAIL", name)
assert all(checks.values())

## 3. Retrieval, scored like the official objective

Each test window's prediction is ranked by cosine similarity against the **unique target embeddings of the test split** (the candidate gallery). Warm-up ranks on pooled `top5_acc`; the sealed phase on `top5_acc_subject_agg`.

In [ ]:
def normalize(v):
    return v / np.maximum(np.linalg.norm(v, axis=-1, keepdims=True), 1e-12)


def evaluate(model, loader):
    preds, targets, eeg = [], [], []
    for X, y, _info in loader:
        preds.append(model.predict(X).cpu().numpy())
        targets.append(y.cpu().numpy())
        eeg.append(X.cpu().numpy())
    preds, targets, eeg = map(np.concatenate, (preds, targets, eeg))
    candidates, target_idx = np.unique(targets, axis=0, return_inverse=True)
    target_idx = target_idx.reshape(-1)
    scores = normalize(preds) @ normalize(candidates).T
    # Rank of the true image among candidates (0 = retrieved first).
    ranks = (scores > scores[np.arange(len(scores)), target_idx][:, None]).sum(1)
    # Simulated data has a single subject, so subject-agg = mean prediction per image.
    keys, pred_sa = group_means(preds, np.stack([np.zeros_like(target_idx), target_idx], 1))
    scores_sa = normalize(pred_sa) @ normalize(candidates).T
    metrics = {
        "top1_acc": topk_accuracy(scores, target_idx, k=1),
        "top5_acc": topk_accuracy(scores, target_idx, k=5),
        "top5_acc_subject_agg": topk_accuracy(scores_sa, keys[:, 1], k=5),
        "chance_top5": 5 / len(candidates),
        "n_candidates": len(candidates),
    }
    return dict(preds=preds, candidates=candidates, target_idx=target_idx,
                scores=scores, ranks=ranks, eeg=eeg, metrics=metrics)


seen = evaluate(model, data["test_loader"])
for k, v in seen["metrics"].items():
    print(f"{k:<22}{v:.3f}" if isinstance(v, float) else f"{k:<22}{v}")

## 4. Visualize

- **Input:** one EEG window as the model receives it (channels × time).
- **Similarity matrix:** cosine similarity between each prediction (rows, sorted by true image) and every candidate (columns). A bright diagonal means the right image scores highest.
- **Rank of the true image:** 0 means retrieved first; top-5 counts ranks 0–4.
- **PCA:** predictions (dots) and candidate embeddings (crosses) in 2-D, colored by image.

In [ ]:
def plot_retrieval(res, title):
    order = np.argsort(res["target_idx"], kind="stable")
    n_cand = len(res["candidates"])
    fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))

    im = axes[0].imshow(res["eeg"][0], aspect="auto", cmap="RdBu_r",
                        extent=[-0.2, 0.8, res["eeg"].shape[1], 0])
    axes[0].set(title="One input window", xlabel="time (s)", ylabel="channel")
    fig.colorbar(im, ax=axes[0])

    im = axes[1].imshow(res["scores"][order], aspect="auto", cmap="viridis")
    axes[1].set(title="Prediction vs candidate similarity", xlabel="candidate image",
                ylabel="test window (sorted by true image)")
    fig.colorbar(im, ax=axes[1])

    axes[2].hist(res["ranks"], bins=np.arange(n_cand + 1) - 0.5, color="#4C72B0")
    axes[2].axvline(4.5, color="k", ls="--", lw=1, label="top-5 cutoff")
    axes[2].set(title=f"Rank of the true image (top-5 = {res['metrics']['top5_acc']:.0%})",
                xlabel="rank (0 = best)", ylabel="windows")
    axes[2].legend()

    both = np.concatenate([normalize(res["preds"]), normalize(res["candidates"])])
    both = both - both.mean(0)
    _, _, vt = np.linalg.svd(both, full_matrices=False)
    xy = both @ vt[:2].T
    n = len(res["preds"])
    axes[3].scatter(*xy[:n].T, c=res["target_idx"], cmap="tab20", s=8, alpha=0.6)
    axes[3].scatter(*xy[n:].T, c=np.arange(n_cand), cmap="tab20", marker="x", s=60)
    axes[3].set(title="PCA: predictions (dots), candidates (x)", xticks=[], yticks=[])

    fig.suptitle(title, fontsize=14)
    fig.tight_layout()
    plt.show()


plot_retrieval(seen, "Images seen during training (same synthetic images as `benchopt run`)")

## 5. Unseen images

The competition tests on **images the model never saw in training**. Here we change the seed, which creates 50 new synthetic images. This model memorized the training images and learned nothing transferable, so expect roughly chance (5/50 = 10% top-5). A real model has to do better than chance on unseen images.

In [ ]:
unseen = evaluate(model, simulated_data(seed="1")["test_loader"])
for k, v in unseen["metrics"].items():
    print(f"{k:<22}{v:.3f}" if isinstance(v, float) else f"{k:<22}{v}")
plot_retrieval(unseen, "New synthetic images (different seed)")

## 6. Benchopt run history

Every `benchopt run` / `make test-submission` writes a results file. The columns match the Codabench scoring keys (`objective_top5_acc` etc.).

In [ ]:
import pandas as pd

runs = []
for f in sorted((TRACK / "outputs").glob("benchopt_run_*.parquet")):
    df = pd.read_parquet(f)
    df["run"] = f.stem.removeprefix("benchopt_run_")
    runs.append(df)
cols = ["run", "solver_name", "dataset_name", "objective_name", "objective_top5_acc", "objective_top1_acc",
        "objective_n_candidates"]
pd.concat(runs)[cols]